# Import pretrained model

In [1]:
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(
    weights='imagenet', # start from filters learned on ImageNet
    include_top=False, # drop the original 1000-class head
    input_shape=(96, 96, 3) # image size going in
)
base.trainable = False  # freeze: training can't change these weights

print(base.output_shape)
print(base.count_params())

c:\Users\Edward\Desktop\python\Lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


(None, 3, 3, 1280)
2257984


# Builing model based on  pretrained model

In [2]:
from tensorflow.keras import layers, Sequential

tl_model = Sequential([
    layers.Input(shape=(32, 32, 3)),            # one CIFAR image
    layers.Resizing(96, 96),                    # 32x32 -> 96x96
    layers.Rescaling(scale=1/127.5, offset=-1), # 0..255 -> -1..1
    base,                                       # frozen MobileNetV2 from before
    layers.GlobalAveragePooling2D(),            # 3x3x1280 -> 1280 numbers
    layers.Dense(10, activation='softmax')      # 10 CIFAR classes
])

tl_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

# Evaluation with early stopping

In [3]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.callbacks import EarlyStopping

#data: raw pixels 0-255 (Rescaling inside the model does the scaling)
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
y_train, y_test = y_train.ravel(), y_test.ravel()   # (N,1) -> (N,)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

tl_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#sanity check on 1000 training images: head is untrained
print(tl_model.evaluate(X_train[:1000], y_train[:1000]))

#train only the head
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

tl_history = tl_model.fit(
    X_train[:10000], y_train[:10000],
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

print(tl_model.evaluate(X_test, y_test))

(50000, 32, 32, 3) (50000,) (10000, 32, 32, 3) (10000,)
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step - accuracy: 0.0620 - loss: 3.2554
[3.255387306213379, 0.06199999898672104]
Epoch 1/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 24s 134ms/step - accuracy: 0.6938 - loss: 0.9241 - val_accuracy: 0.8040 - val_loss: 0.5840
Epoch 2/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 19s 132ms/step - accuracy: 0.8371 - loss: 0.4768 - val_accuracy: 0.8310 - val_loss: 0.4976
Epoch 3/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 18s 127ms/step - accuracy: 0.8624 - loss: 0.3996 - val_accuracy: 0.8450 - val_loss: 0.4668
Epoch 4/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 18s 131ms/step - accuracy: 0.8812 - loss: 0.3480 - val_accuracy: 0.8450 - val_loss: 0.4514
Epoch 5/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 18s 128ms/step - accuracy: 0.8974 - loss: 0.3060 - val_accuracy: 0.8550 - val_loss: 0.4359
Epoch 6/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 18s 129ms/step - accuracy: 0.9113 - loss: 0.2774 - val_accuracy: 0.8480 - val_loss: 0.4489
Epoch 7/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 22s

# Fine tunning

In [4]:
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# 1) unfreeze the SAME base that is inside tl_model
base.trainable = True

# 2) re-freeze all but the last 20 layers
fine_tune_at = 20
for layer in base.layers[:-fine_tune_at]:
    layer.trainable = False

# 3) keep BatchNormalization layers frozen
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# 4) recompile with a tiny learning rate (required after changing trainable)
tl_model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
tl_model.summary()

# 5) train again
early_stop_ft = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
tl_history_ft = tl_model.fit(
    X_train[:10000], y_train[:10000],
    epochs=10, batch_size=64, validation_split=0.1,
    callbacks=[early_stop_ft]
)

# 6) final test
print(tl_model.evaluate(X_test, y_test))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 1,207,690 (4.61 MB)

 Non-trainable params: 1,063,104 (4.06 MB)

Epoch 1/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 29s 156ms/step - accuracy: 0.9132 - loss: 0.2554 - val_accuracy: 0.8580 - val_loss: 0.4336
Epoch 2/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 22s 154ms/step - accuracy: 0.9311 - loss: 0.2118 - val_accuracy: 0.8530 - val_loss: 0.4367
Epoch 3/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 20s 144ms/step - accuracy: 0.9433 - loss: 0.1824 - val_accuracy: 0.8620 - val_loss: 0.4428
313/313 ━━━━━━━━━━━━━━━━━━━━ 20s 64ms/step - accuracy: 0.8447 - loss: 0.4626
[0.4626154899597168, 0.8446999788284302]
